## 1. Create the client and request a tool call

Load the API key, create an OpenAI client, and ask the model for the stock price. The model may return one or more function calls.

In [125]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

response = client.responses.create(
    model="gpt-5.6-sol",
    input = "what is the stock price of Google?",
    tools=my_tools
    )
print(response.output)



[ResponseReasoningItem(id='rs_03913a8ee726de28006aa6c64e260487d2acce7c31e65e485e', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqpsZPgrJsBiAQpA8A3PQ8TBg0Gtw2AV0bSijUvVLEzB9scHoep1zV22TPpp9wKWvdG39amHwGrN1oZVAiWgFJSIjHt1TGcooUm1HqiO4B7SS7FzCisoZS1G5syCiJ8vINS9M61v9IpnraeW1xpFZWIpKgcl4gWVgkjS-WlFrnpgNjy0bcCxCaYz7urnAXr5lD_TB_9NN7Ji66-2EAGxXQbcDNW5lX_eRoBfONnp2XYuJYTxHddu49rEbxn_rNyLsPKfwQ8hzA_XFNNoziCMGU6iTGXjjLInyB32Ccooe2KqPQxBTyCJVCCrcCQYS8pf6b_v9DCb_bN8J09PTYceyY3BTA5vdYF59Zt-0jetHP6FkPp1RdKZPt4KAiCqTcjJcfG0-R6kj_fPAic6MtKyXb-oxYA5bp7nkhb5pNufLLVUPep0whAJggWCQ8SHfdhov-PhTnhuzbZheChIH86tSWVUa_-64TFiBGrc95H-VdZMEdhnu91oojfXJ2d6Iu19Pw10nhANcNUTqDD2lpzjReTg2aI85pgtTJ44Y2xRhqGdfR5qZZ0A8Z6E2sBhSTJJlZOqUEw6B5f55R69OUllRUU4ePY2nVaXA9BYrKJxXXXUgcSABs3d895BHTxa_tTI8grFKjGmcBh_kwNlJVPU0EP70wzVp0X6gYfpvfx_mfOi3xnhc1Fx4q8db_P7PA0_e7XloOiHwIEIusUxDSX9qiT08WqU8OALM66xiLS45kdWRy6bQio_Ofcxom-9iKH7AftQm4CsM0B1iz2Ov69GFl9qD-pRG3TXRIp25GSGV2tIO25fdU5U3zeed6Y3X8Zv24_w1XIGaCh6LqMI_M

## 2. Save the response ID

Store the response ID so the tool results can be submitted as a continuation of this exact model response.

In [126]:
response_id = response.id
response_id

'resp_03913a8ee726de28006aa6c64d496487d2a7ad3f6bdb84be29'

## 3. Define the local stock-price function

This local Python function stands in for a real stock-price service and returns a value for a ticker symbol.

In [127]:
def get_stock_price(ticker):
    prices = {
        "AAPL": 150.25,
        "MSFT": 300.50,
        "GOOGL": 2800.75
    }
    return prices.get(ticker.upper(), "Ticker not found")

## 4. Describe the function tool to the model

Define the tool name, purpose, input schema, and required ticker argument that the model can use when deciding whether to call the function.

In [128]:
my_tools = [
    {
        "type": "function",
        "name": "get_stock_price", 
        "description": "Get the stock price of a given ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The ticker symbol of the stock."
                }
            },
            "required": ["ticker"]
        }
    }
]

## 5. Extract every function call

Responses can include reasoning items and multiple function calls. Select only function-call items and keep each call's `call_id` for the matching result.

In [129]:
function_calls = [item for item in response.output if item.type == "function_call"]
if not function_calls:
    raise RuntimeError("The response did not contain a function call.")

tool_call = function_calls[0]
tool_call
args = json.loads(tool_call.arguments)
function_name = tool_call.name
call_id = tool_call.call_id
args

{'ticker': 'GOOGL'}

## 6. Parse the model's arguments

Read the JSON arguments and the function name from the selected tool call. Use `call_id`, not the output item's separate `id`, when returning results.

In [130]:
if function_name == "get_stock_price":
    stock_price = get_stock_price(args["ticker"])
stock_price

2800.75

## 7. Execute the requested function

Use the parsed ticker to call the local function and obtain the tool result.

In [131]:
tool_output = []
for tool_call in function_calls:
    call_args = json.loads(tool_call.arguments)
    if tool_call.name == "get_stock_price":
        result = get_stock_price(call_args["ticker"])
    else:
        result = {"error": f"Unknown function: {tool_call.name}"}

    tool_output.append({
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps({"stock_price": result})
    })

## 8. Build one result for each tool call

Create a `function_call_output` item for every function call, preserving each matching `call_id`. This is required when the model makes parallel calls.

In [132]:
response = client.responses.create(
    model="gpt-5.6-sol",
    input=tool_output,
    previous_response_id=response_id,
    tools=my_tools)
response.output_text

'Alphabet (Google), ticker **GOOGL**, is trading at **$2,800.75** per share.'

## 9. Continue the conversation with the tool results

Send the function outputs with the original response ID so the model can use them to produce the final natural-language answer.